In [1]:
import os
import sys
import duckdb
import s3fs
from dotenv import load_dotenv
from pathlib import Path
import pandas as pd
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

fs = s3fs.S3FileSystem(
    key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret=os.getenv("AWS_SECRET_ACCESS_KEY"),
    client_kwargs={"region_name": os.getenv("AWS_REGION")},
)
bucket = os.getenv("S3_BUCKET_RAW")
con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))

## fastf1_results data preview raw -> staging

In [32]:
with fs.open(f"s3://{bucket}/fastf1/2026/1/results.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)
# df_results[["driver_id", "country_code"]]

,driver_number,broadcast_name,abbreviation,driver_id,team_name,team_color,team_id,first_name,last_name,full_name,headshot_url,country_code,position,classified_position,grid_position,q1,q2,q3,time,status,points,laps,season,round,source,ingested_at
0,63,G RUSSELL,RUS,russell,Mercedes,00D7B6,mercedes,George,Russell,George Russell,https://media.formula1.com/d_driver_fallback_i...,,1.0,1,1.0,NaT,NaT,NaT,0 days 01:23:06.801000,Finished,25.0,58.0,2026,1,fastf1,2026-03-22T03:06:36.278058+00:00
1,12,K ANTONELLI,ANT,antonelli,Mercedes,00D7B6,mercedes,Kimi,Antonelli,Kimi Antonelli,https://media.formula1.com/d_driver_fallback_i...,,2.0,2,2.0,NaT,NaT,NaT,0 days 00:00:02.974000,Finished,18.0,58.0,2026,1,fastf1,2026-03-22T03:06:36.278058+00:00
2,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,https://media.formula1.com/d_driver_fallback_i...,,3.0,3,4.0,NaT,NaT,NaT,0 days 00:00:15.519000,Finished,15.0,58.0,2026,1,fastf1,2026-03-22T03:06:36.278058+00:00


In [33]:
df = con.execute("SELECT * FROM bronze.fastf1_results").df()
df.head(3)
# df[["driver_id", "country_code"]]

,driver_number,broadcast_name,abbreviation,driver_id,team_name,team_color,team_id,first_name,last_name,full_name,headshot_url,country_code,position,classified_position,grid_position,q1,q2,q3,time,status,points,laps,season,round,source,ingested_at
0,63,G RUSSELL,RUS,russell,Mercedes,00D7B6,mercedes,George,Russell,George Russell,https://media.formula1.com/d_driver_fallback_i...,,1.0,1,1.0,<NA>,<NA>,<NA>,4986801000000,Finished,25.0,58.0,2026,1,fastf1,2026-03-22T03:06:36.278058+00:00
1,12,K ANTONELLI,ANT,antonelli,Mercedes,00D7B6,mercedes,Kimi,Antonelli,Kimi Antonelli,https://media.formula1.com/d_driver_fallback_i...,,2.0,2,2.0,<NA>,<NA>,<NA>,2974000000,Finished,18.0,58.0,2026,1,fastf1,2026-03-22T03:06:36.278058+00:00
2,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,https://media.formula1.com/d_driver_fallback_i...,,3.0,3,4.0,<NA>,<NA>,<NA>,15519000000,Finished,15.0,58.0,2026,1,fastf1,2026-03-22T03:06:36.278058+00:00


In [3]:
df = con.execute("SELECT * FROM silver.fastf1_results").df()
pd.set_option('display.max_columns', None)
df.head(3)
# df.dtypes

,driver_number,broadcast_name,abbreviation,driver_id,team_name,team_color,team_id,first_name,last_name,full_name,position,classified_position,grid_position,status,points,laps,season,round,source,ingested_at,time_seconds
0,63,G RUSSELL,RUS,russell,Mercedes,00D7B6,mercedes,George,Russell,George Russell,1,1,1,Finished,25.0,58,2026,1,fastf1,2026-03-21 17:06:36.278058-10:00,4986.801
1,12,K ANTONELLI,ANT,antonelli,Mercedes,00D7B6,mercedes,Kimi,Antonelli,Kimi Antonelli,2,2,2,Finished,18.0,58,2026,1,fastf1,2026-03-21 17:06:36.278058-10:00,2.974
2,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,3,3,4,Finished,15.0,58,2026,1,fastf1,2026-03-21 17:06:36.278058-10:00,15.519


In [47]:
con.execute("SELECT * FROM staging.stg_fastf1_results").df().head(3)

,season,round,driver_id,driver_number,broadcast_name,abbreviation,first_name,last_name,full_name,team_id,team_name,team_color,grid_position,finish_position,classified_position,laps,status,points,race_time_seconds
0,2026,1,russell,63,G RUSSELL,RUS,George,Russell,George Russell,mercedes,Mercedes,00D7B6,1,1,1,58,Finished,25.0,4986.801
1,2026,1,antonelli,12,K ANTONELLI,ANT,Kimi,Antonelli,Kimi Antonelli,mercedes,Mercedes,00D7B6,2,2,2,58,Finished,18.0,2.974
2,2026,1,leclerc,16,C LECLERC,LEC,Charles,Leclerc,Charles Leclerc,ferrari,Ferrari,ED1131,4,3,3,58,Finished,15.0,15.519


## laps data preview raw -> staging

In [11]:
with fs.open(f"s3://{bucket}/fastf1/2026/1/laps.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,time,driver,driver_number,lap_time,lap_number,stint,pit_out_time,pit_in_time,sector1_time,sector2_time,sector3_time,sector1_session_time,sector2_session_time,sector3_session_time,speed_i1,speed_i2,speed_fl,speed_st,is_personal_best,compound,tyre_life,fresh_tyre,team,lap_start_time,lap_start_date,track_status,position,deleted,deleted_reason,fast_f1_generated,is_accurate,season,round,source,ingested_at
0,0 days 01:03:56.437000,NOR,1,0 days 00:01:36.458000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:18.163000,0 days 00:00:38.796000,NaT,0 days 01:03:17.998000,0 days 01:03:56.692000,229.0,291.0,304.0,217.0,False,MEDIUM,1.0,True,McLaren,0 days 01:02:19.743000,NaT,1,6.0,None,,False,False,2026,1,fastf1,2026-03-22T03:06:37.399936+00:00
1,0 days 01:05:23.781000,NOR,1,0 days 00:01:27.344000,2.0,1.0,NaT,NaT,0 days 00:00:31.074000,0 days 00:00:18.116000,0 days 00:00:38.154000,0 days 01:04:27.511000,0 days 01:04:45.627000,0 days 01:05:23.781000,243.0,287.0,303.0,260.0,True,MEDIUM,2.0,True,McLaren,0 days 01:03:56.437000,NaT,1,6.0,None,,False,True,2026,1,fastf1,2026-03-22T03:06:37.399936+00:00
2,0 days 01:06:50.644000,NOR,1,0 days 00:01:26.863000,3.0,1.0,NaT,NaT,0 days 00:00:30.541000,0 days 00:00:18.252000,0 days 00:00:38.070000,0 days 01:05:54.322000,0 days 01:06:12.574000,0 days 01:06:50.644000,245.0,311.0,286.0,273.0,True,MEDIUM,3.0,True,McLaren,0 days 01:05:23.781000,NaT,1,7.0,None,,False,True,2026,1,fastf1,2026-03-22T03:06:37.399936+00:00


In [15]:
df = con.execute("SELECT * FROM bronze.fastf1_laps").df()
df.head(3)

,time,driver,driver_number,lap_time,lap_number,stint,pit_out_time,pit_in_time,sector1_time,sector2_time,sector3_time,sector1_session_time,sector2_session_time,sector3_session_time,speed_i1,speed_i2,speed_fl,speed_st,is_personal_best,compound,tyre_life,fresh_tyre,team,lap_start_time,lap_start_date,track_status,position,deleted,deleted_reason,fast_f1_generated,is_accurate,season,round,source,ingested_at
0,3836437000000,NOR,1,96458000000,1.0,1.0,<NA>,<NA>,<NA>,18163000000,38796000000,<NA>,3797998000000,3836692000000,229.0,291.0,304.0,217.0,False,MEDIUM,1.0,True,McLaren,3739743000000,NaT,1,6.0,<NA>,,False,False,2026,1,fastf1,2026-03-22T03:06:37.399936+00:00
1,3923781000000,NOR,1,87344000000,2.0,1.0,<NA>,<NA>,31074000000,18116000000,38154000000,3867511000000,3885627000000,3923781000000,243.0,287.0,303.0,260.0,True,MEDIUM,2.0,True,McLaren,3836437000000,NaT,1,6.0,<NA>,,False,True,2026,1,fastf1,2026-03-22T03:06:37.399936+00:00
2,4010644000000,NOR,1,86863000000,3.0,1.0,<NA>,<NA>,30541000000,18252000000,38070000000,3954322000000,3972574000000,4010644000000,245.0,311.0,286.0,273.0,True,MEDIUM,3.0,True,McLaren,3923781000000,NaT,1,7.0,<NA>,,False,True,2026,1,fastf1,2026-03-22T03:06:37.399936+00:00


In [41]:
df = con.execute("SELECT * FROM silver.fastf1_laps").df()
pd.set_option('display.max_columns', None)
df.head(3)
# df.dtypes

,driver,driver_number,lap_number,stint,speed_i1,speed_i2,speed_fl,speed_st,is_personal_best,compound,tyre_life,fresh_tyre,team,lap_start_date,track_status,position,deleted,deleted_reason,fast_f1_generated,is_accurate,season,round,source,ingested_at,time_seconds,lap_time_seconds,pit_out_time_seconds,pit_in_time_seconds,sector1_time_seconds,sector2_time_seconds,sector3_time_seconds,sector1_session_time_seconds,sector2_session_time_seconds,sector3_session_time_seconds,lap_start_time_seconds
0,nor,1,1,1,229.0,291.0,304.0,217.0,False,MEDIUM,1,True,mclaren,NaT,1,6,<NA>,<NA>,False,False,2026,1,fastf1,2026-03-21 17:06:37.399936-10:00,3836.437,96.458,NaN,NaN,NaN,18.163,38.796,NaN,3797.998,3836.692,3739.743
1,nor,1,2,1,243.0,287.0,303.0,260.0,True,MEDIUM,2,True,mclaren,NaT,1,6,<NA>,<NA>,False,True,2026,1,fastf1,2026-03-21 17:06:37.399936-10:00,3923.781,87.344,NaN,NaN,31.074,18.116,38.154,3867.511,3885.627,3923.781,3836.437
2,nor,1,3,1,245.0,311.0,286.0,273.0,True,MEDIUM,3,True,mclaren,NaT,1,7,<NA>,<NA>,False,True,2026,1,fastf1,2026-03-21 17:06:37.399936-10:00,4010.644,86.863,NaN,NaN,30.541,18.252,38.070,3954.322,3972.574,4010.644,3923.781


In [48]:
df = con.execute("SELECT * FROM staging.stg_fastf1_laps").df()
# df.dtypes
df.head(3)

,season,round,abbreviation,driver_number,team,lap_number,stint,lap_time_seconds,sector1_time_seconds,sector2_time_seconds,sector3_time_seconds,session_elapsed_seconds,lap_start_time_seconds,pit_out_time_seconds,pit_in_time_seconds,sector1_session_time_seconds,sector2_session_time_seconds,sector3_session_time_seconds,speed_i1,speed_i2,speed_fl,speed_st,compound,tyre_life,fresh_tyre,is_personal_best,is_accurate,fast_f1_generated,track_status,position
0,2026,1,NOR,1,mclaren,1,1,96.458,NaN,18.163,38.796,3836.437,3739.743,NaN,NaN,NaN,3797.998,3836.692,229.0,291.0,304.0,217.0,MEDIUM,1,True,False,False,False,1,6
1,2026,1,NOR,1,mclaren,2,1,87.344,31.074,18.116,38.154,3923.781,3836.437,NaN,NaN,3867.511,3885.627,3923.781,243.0,287.0,303.0,260.0,MEDIUM,2,True,True,True,False,1,6
2,2026,1,NOR,1,mclaren,3,1,86.863,30.541,18.252,38.070,4010.644,3923.781,NaN,NaN,3954.322,3972.574,4010.644,245.0,311.0,286.0,273.0,MEDIUM,3,True,True,True,False,1,7


## telemetry data preview raw -> staging

In [43]:
with fs.open(f"s3://{bucket}/fastf1/2026/1/telemetry.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,date,session_time,driver_ahead,distance_to_driver_ahead,time,rpm,speed,n_gear,throttle,brake,drs,source,distance,relative_distance,status,x,y,z,season,round,driver,team,lap_number,ingested_at
0,2026-03-08 04:03:26.366,0 days 01:02:19.743000,,0.0,0 days 00:00:00,12500.04373,0.0,1,24.0,True,0,fastf1,-0.001395,-2.633795e-07,OnTrack,-768.005561,-1739.998651,88.000001,2026,1,NOR,McLaren,1,2026-03-22T03:12:16.206831+00:00
1,2026-03-08 04:03:26.412,0 days 01:02:19.789000,,0.0,0 days 00:00:00.046000,12493.00000,0.0,1,24.0,True,0,fastf1,0.000000,0.000000e+00,OnTrack,-768.036195,-1739.991218,88.000005,2026,1,NOR,McLaren,1,2026-03-22T03:12:16.206831+00:00
2,2026-03-08 04:03:26.572,0 days 01:02:19.949000,,0.0,0 days 00:00:00.206000,12494.00000,0.0,1,24.0,True,0,fastf1,0.000000,0.000000e+00,OnTrack,-768.064084,-1739.984451,88.000009,2026,1,NOR,McLaren,1,2026-03-22T03:12:16.206831+00:00


In [44]:
df = con.execute("SELECT * FROM bronze.fastf1_telemetry").df()
df.head(3)

,date,session_time,driver_ahead,distance_to_driver_ahead,time,rpm,speed,n_gear,throttle,brake,drs,source,distance,relative_distance,status,x,y,z,season,round,driver,team,lap_number,ingested_at
0,2026-03-08 04:03:26.366,3739743000000,,0.0,0,12500.04373,0.0,1,24.0,True,0,fastf1,-0.001395,-2.633795e-07,OnTrack,-768.005561,-1739.998651,88.000001,2026,1,NOR,McLaren,1,2026-03-22T03:12:16.206831+00:00
1,2026-03-08 04:03:26.412,3739789000000,,0.0,46000000,12493.00000,0.0,1,24.0,True,0,fastf1,0.000000,0.000000e+00,OnTrack,-768.036195,-1739.991218,88.000005,2026,1,NOR,McLaren,1,2026-03-22T03:12:16.206831+00:00
2,2026-03-08 04:03:26.572,3739949000000,,0.0,206000000,12494.00000,0.0,1,24.0,True,0,fastf1,0.000000,0.000000e+00,OnTrack,-768.064084,-1739.984451,88.000009,2026,1,NOR,McLaren,1,2026-03-22T03:12:16.206831+00:00


In [3]:
df = con.execute("SELECT * FROM silver.fastf1_telemetry").df()
pd.set_option('display.max_columns', None)
df.head(3)
# df.dtypes

,date,driver_ahead,distance_to_driver_ahead,rpm,speed,n_gear,throttle,brake,drs,source,distance,relative_distance,status,x,y,z,season,round,driver,team,lap_number,ingested_at,session_time_seconds,lap_time_seconds
0,2026-03-07 18:03:26.366000-10:00,None,0.0,12500.04373,0.0,1,24.0,True,0,fastf1,-0.001395,-2.633795e-07,OnTrack,-768.005561,-1739.998651,88.000001,2026,1,nor,mclaren,1,2026-03-21 17:12:16.206831-10:00,3739.743,0.000
1,2026-03-07 18:03:26.412000-10:00,None,0.0,12493.00000,0.0,1,24.0,True,0,fastf1,0.000000,0.000000e+00,OnTrack,-768.036195,-1739.991218,88.000005,2026,1,nor,mclaren,1,2026-03-21 17:12:16.206831-10:00,3739.789,0.046
2,2026-03-07 18:03:26.572000-10:00,None,0.0,12494.00000,0.0,1,24.0,True,0,fastf1,0.000000,0.000000e+00,OnTrack,-768.064084,-1739.984451,88.000009,2026,1,nor,mclaren,1,2026-03-21 17:12:16.206831-10:00,3739.949,0.206


In [49]:
df = con.execute("SELECT * FROM staging.stg_fastf1_telemetry").df()
# df.dtypes
df.head(5)

,season,round,abbreviation,team,lap_number,telemetry_timestamp,session_elapsed_seconds,lap_time_seconds,rpm,speed,n_gear,throttle,brake,drs,distance,relative_distance,x,y,z,driver_ahead,distance_to_driver_ahead,track_status
0,2026,1,NOR,mclaren,1,2026-03-07 18:03:26.366000-10:00,3739.743,0.000,12500.043730,0.0,1,24.0,True,0,-0.001395,-2.633795e-07,-768.005561,-1739.998651,88.000001,None,0.0,OnTrack
1,2026,1,NOR,mclaren,1,2026-03-07 18:03:26.412000-10:00,3739.789,0.046,12493.000000,0.0,1,24.0,True,0,0.000000,0.000000e+00,-768.036195,-1739.991218,88.000005,None,0.0,OnTrack
2,2026,1,NOR,mclaren,1,2026-03-07 18:03:26.572000-10:00,3739.949,0.206,12494.000000,0.0,1,24.0,True,0,0.000000,0.000000e+00,-768.064084,-1739.984451,88.000009,None,0.0,OnTrack
3,2026,1,NOR,mclaren,1,2026-03-07 18:03:26.698000-10:00,3740.075,0.332,12498.409996,0.0,1,24.0,True,0,-0.031676,-5.980914e-06,-768.000000,-1740.000000,88.000000,None,0.0,OnTrack
4,2026,1,NOR,mclaren,1,2026-03-07 18:03:26.772000-10:00,3740.149,0.406,12501.000000,0.0,1,24.0,True,0,0.000000,0.000000e+00,-767.927023,-1740.017707,87.999990,None,0.0,OnTrack


In [3]:
print(con.execute("""
    SELECT round, COUNT(*) as rows
    FROM staging.stg_fastf1_telemetry
    WHERE season = 2026
    GROUP BY round ORDER BY round
""").df())

   round    rows
0      1  684078
1      2  706701


In [4]:
print(con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'staging'
      AND table_name = 'stg_fastf1_telemetry'
    ORDER BY ordinal_position
""").df().to_string())

                 column_name                 data_type
0                     season                    BIGINT
1                      round                    BIGINT
2               abbreviation                   VARCHAR
3                       team                   VARCHAR
4                 lap_number                    BIGINT
5        telemetry_timestamp  TIMESTAMP WITH TIME ZONE
6    session_elapsed_seconds                    DOUBLE
7           lap_time_seconds                    DOUBLE
8                        rpm                    DOUBLE
9                      speed                    DOUBLE
10                    n_gear                    BIGINT
11                  throttle                    DOUBLE
12                     brake                   BOOLEAN
13                       drs                    BIGINT
14                  distance                    DOUBLE
15         relative_distance                    DOUBLE
16                         x                    DOUBLE
17        

In [5]:
print(con.execute("""
    SELECT
        COUNT(*)                                                AS total_rows,
        SUM(CASE WHEN driver_ahead IS NULL THEN 1 END)         AS driver_ahead_nulls,
        SUM(CASE WHEN distance_to_driver_ahead = 0
                  OR distance_to_driver_ahead IS NULL
                 THEN 1 END)                                   AS dist_ahead_sparse,
        SUM(CASE WHEN x IS NULL THEN 1 END)                    AS x_nulls,
        SUM(CASE WHEN drs IS NULL THEN 1 END)                  AS drs_nulls
    FROM staging.stg_fastf1_telemetry
    WHERE season = 2026
""").df())

   total_rows  driver_ahead_nulls  dist_ahead_sparse  x_nulls  drs_nulls
0     1390779             87779.0            60738.0      NaN        NaN


## fast_weather data preview raw -> staging

In [12]:
with fs.open(f"s3://{bucket}/fastf1/2026/1/weather.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,time,air_temp,humidity,pressure,rainfall,track_temp,wind_direction,wind_speed,season,round,source,ingested_at
0,0 days 00:00:13.466000,23.1,55.8,1013.7,False,36.2,114,1.8,2026,1,fastf1,2026-03-22T03:06:38.002261+00:00
1,0 days 00:01:13.482000,23.2,55.1,1013.6,False,35.9,110,3.0,2026,1,fastf1,2026-03-22T03:06:38.002261+00:00
2,0 days 00:02:13.483000,23.2,55.2,1013.6,False,35.6,115,3.2,2026,1,fastf1,2026-03-22T03:06:38.002261+00:00


In [14]:
df = con.execute("SELECT * FROM bronze.fastf1_weather").df()
df.head(3)

,time,air_temp,humidity,pressure,rainfall,track_temp,wind_direction,wind_speed,season,round,source,ingested_at
0,13466000000,23.1,55.8,1013.7,False,36.2,114,1.8,2026,1,fastf1,2026-03-22T03:06:38.002261+00:00
1,73482000000,23.2,55.1,1013.6,False,35.9,110,3.0,2026,1,fastf1,2026-03-22T03:06:38.002261+00:00
2,133483000000,23.2,55.2,1013.6,False,35.6,115,3.2,2026,1,fastf1,2026-03-22T03:06:38.002261+00:00


In [15]:
df = con.execute("SELECT * FROM silver.fastf1_weather").df()
df.head(3)

,air_temp,humidity,pressure,rainfall,track_temp,wind_direction,wind_speed,season,round,source,ingested_at,time_seconds
0,23.1,55.8,1013.7,False,36.2,114,1.8,2026,1,fastf1,2026-03-21 17:06:38.002261-10:00,13.466
1,23.2,55.1,1013.6,False,35.9,110,3.0,2026,1,fastf1,2026-03-21 17:06:38.002261-10:00,73.482
2,23.2,55.2,1013.6,False,35.6,115,3.2,2026,1,fastf1,2026-03-21 17:06:38.002261-10:00,133.483


In [50]:
df = con.execute("SELECT * FROM staging.stg_fastf1_weather").df()
# df.dtypes
df.head(5)

,season,round,session_elapsed_seconds,air_temp,track_temp,humidity,pressure,rainfall,wind_direction,wind_speed
0,2026,1,13.466,23.1,36.2,55.8,1013.7,False,114,1.8
1,2026,1,73.482,23.2,35.9,55.1,1013.6,False,110,3.0
2,2026,1,133.483,23.2,35.6,55.2,1013.6,False,115,3.2
3,2026,1,193.485,23.1,35.7,55.7,1013.7,False,104,3.4
4,2026,1,253.518,23.1,35.8,55.8,1013.8,False,108,3.1


## jolpica driver_standings data preview raw -> staging

In [17]:
with fs.open(f"s3://{bucket}/jolpica/2026/driver_standings.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,season,round,driver_id,driver_code,driver_name,driver_dob,driver_country,constructor,constructor_country,position,points,wins,source,ingested_at
0,2026,2,russell,RUS,George Russell,1998-02-15,British,Mercedes,German,1,51.0,1,jolpica,2026-03-18T21:45:04.180898+00:00
1,2026,2,antonelli,ANT,Andrea Kimi Antonelli,2006-08-25,Italian,Mercedes,German,2,47.0,1,jolpica,2026-03-18T21:45:04.180898+00:00
2,2026,2,leclerc,LEC,Charles Leclerc,1997-10-16,Monegasque,Ferrari,Italian,3,34.0,0,jolpica,2026-03-18T21:45:04.180898+00:00


In [18]:
df = con.execute("SELECT * FROM bronze.jolpica_driver_standings").df()
df.head(3)

,season,round,driver_id,driver_code,driver_name,driver_dob,driver_country,constructor,constructor_country,position,points,wins,source,ingested_at
0,2026,2,russell,RUS,George Russell,1998-02-15,British,Mercedes,German,1,51.0,1,jolpica,2026-03-18T21:45:04.180898+00:00
1,2026,2,antonelli,ANT,Andrea Kimi Antonelli,2006-08-25,Italian,Mercedes,German,2,47.0,1,jolpica,2026-03-18T21:45:04.180898+00:00
2,2026,2,leclerc,LEC,Charles Leclerc,1997-10-16,Monegasque,Ferrari,Italian,3,34.0,0,jolpica,2026-03-18T21:45:04.180898+00:00


In [19]:
df = con.execute("SELECT * FROM silver.jolpica_driver_standings").df()
df.head(3)

,season,round,driver_id,driver_code,driver_name,driver_dob,driver_country,constructor,constructor_country,position,points,wins,source,ingested_at
0,2026,2,russell,RUS,George Russell,1998-02-15,British,mercedes,German,1,51.0,1,jolpica,2026-03-18 11:45:04.180898-10:00
1,2026,2,antonelli,ANT,Andrea Kimi Antonelli,2006-08-25,Italian,mercedes,German,2,47.0,1,jolpica,2026-03-18 11:45:04.180898-10:00
2,2026,2,leclerc,LEC,Charles Leclerc,1997-10-16,Monegasque,ferrari,Italian,3,34.0,0,jolpica,2026-03-18 11:45:04.180898-10:00


In [51]:
df = con.execute("SELECT * FROM staging.stg_jolpica_driver_standings").df()
# df.dtypes
df.head(5)

,season,round,driver_id,abbreviation,driver_name,driver_dob,driver_country,constructor,constructor_country,championship_position,championship_points,wins
0,2026,2,russell,RUS,George Russell,1998-02-15,British,mercedes,German,1,51.0,1
1,2026,2,antonelli,ANT,Andrea Kimi Antonelli,2006-08-25,Italian,mercedes,German,2,47.0,1
2,2026,2,leclerc,LEC,Charles Leclerc,1997-10-16,Monegasque,ferrari,Italian,3,34.0,0
3,2026,2,hamilton,HAM,Lewis Hamilton,1985-01-07,British,ferrari,Italian,4,33.0,0
4,2026,2,bearman,BEA,Oliver Bearman,2005-05-08,British,haas_f1_team,American,5,17.0,0


## jolpica constructor_standings data preview raw -> staging

In [27]:
with fs.open(f"s3://{bucket}/jolpica/2026/constructor_standings.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,season,round,constructor_id,constructor,constructor_country,position,points,wins,source,ingested_at
0,2026,2,mercedes,Mercedes,German,1,98.0,2,jolpica,2026-03-18T21:45:45.373097+00:00
1,2026,2,ferrari,Ferrari,Italian,2,67.0,0,jolpica,2026-03-18T21:45:45.373097+00:00
2,2026,2,mclaren,McLaren,British,3,18.0,0,jolpica,2026-03-18T21:45:45.373097+00:00


In [28]:
df = con.execute("SELECT * FROM bronze.jolpica_constructor_standings").df()
df.head(3)

,season,round,constructor_id,constructor,constructor_country,position,points,wins,source,ingested_at
0,2026,2,mercedes,Mercedes,German,1,98.0,2,jolpica,2026-03-18T21:45:45.373097+00:00
1,2026,2,ferrari,Ferrari,Italian,2,67.0,0,jolpica,2026-03-18T21:45:45.373097+00:00
2,2026,2,mclaren,McLaren,British,3,18.0,0,jolpica,2026-03-18T21:45:45.373097+00:00


In [29]:
df = con.execute("SELECT * FROM silver.jolpica_constructor_standings").df()
df.head(3)

,season,round,constructor_id,constructor,constructor_country,position,points,wins,source,ingested_at
0,2026,2,mercedes,Mercedes,German,1,98.0,2,jolpica,2026-03-18 11:45:45.373097-10:00
1,2026,2,ferrari,Ferrari,Italian,2,67.0,0,jolpica,2026-03-18 11:45:45.373097-10:00
2,2026,2,mclaren,McLaren,British,3,18.0,0,jolpica,2026-03-18 11:45:45.373097-10:00


In [52]:
df = con.execute("SELECT * FROM staging.stg_jolpica_constructor_standings").df()
# df.dtypes
df.head(5)

,season,round,constructor_id,constructor,constructor_country,championship_position,championship_points,wins
0,2026,2,mercedes,Mercedes,German,1,98.0,2
1,2026,2,ferrari,Ferrari,Italian,2,67.0,0
2,2026,2,mclaren,McLaren,British,3,18.0,0
3,2026,2,haas,Haas F1 Team,American,4,17.0,0
4,2026,2,red_bull,Red Bull,Austrian,5,12.0,0


## jolpica race_results data preview raw -> staging

In [31]:
with fs.open(f"s3://{bucket}/jolpica/2026/race_results.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,season,round,race_name,circuit,date,driver_id,driver_code,driver_name,constructor,grid,position,points,status,fastest_lap_rank,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,russell,RUS,George Russell,Mercedes,1,1,25.0,Finished,6.0,jolpica,2026-03-22T09:05:35.989350+00:00
1,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,antonelli,ANT,Andrea Kimi Antonelli,Mercedes,2,2,18.0,Finished,3.0,jolpica,2026-03-22T09:05:35.989350+00:00
2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,leclerc,LEC,Charles Leclerc,Ferrari,4,3,15.0,Finished,5.0,jolpica,2026-03-22T09:05:35.989350+00:00


In [32]:
df = con.execute("SELECT * FROM bronze.jolpica_race_results").df()
df.head(3)

,season,round,race_name,circuit,date,driver_id,driver_code,driver_name,constructor,grid,position,points,status,fastest_lap_rank,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,russell,RUS,George Russell,Mercedes,1,1,25.0,Finished,6.0,jolpica,2026-03-22T09:05:35.989350+00:00
1,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,antonelli,ANT,Andrea Kimi Antonelli,Mercedes,2,2,18.0,Finished,3.0,jolpica,2026-03-22T09:05:35.989350+00:00
2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,leclerc,LEC,Charles Leclerc,Ferrari,4,3,15.0,Finished,5.0,jolpica,2026-03-22T09:05:35.989350+00:00


In [33]:
df = con.execute("SELECT * FROM silver.jolpica_race_results").df()
df.head(3)

,season,round,race_name,circuit,date,driver_id,driver_code,driver_name,constructor,grid,position,points,status,fastest_lap_rank,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,russell,RUS,George Russell,mercedes,1,1,25.0,Finished,6,jolpica,2026-03-21 23:05:35.989350-10:00
1,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,antonelli,ANT,Andrea Kimi Antonelli,mercedes,2,2,18.0,Finished,3,jolpica,2026-03-21 23:05:35.989350-10:00
2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,leclerc,LEC,Charles Leclerc,ferrari,4,3,15.0,Finished,5,jolpica,2026-03-21 23:05:35.989350-10:00


In [53]:
df = con.execute("SELECT * FROM staging.stg_jolpica_race_results").df()
# df.dtypes
df.head(5)

,season,round,driver_id,abbreviation,driver_name,constructor,race_name,circuit,race_date,grid_position,finish_position,points,status,fastest_lap_rank
0,2026,1,russell,RUS,George Russell,mercedes,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,1,25.0,Finished,6
1,2026,1,antonelli,ANT,Andrea Kimi Antonelli,mercedes,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,2,2,18.0,Finished,3
2,2026,1,leclerc,LEC,Charles Leclerc,ferrari,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,4,3,15.0,Finished,5
3,2026,1,hamilton,HAM,Lewis Hamilton,ferrari,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,7,4,12.0,Finished,4
4,2026,1,norris,NOR,Lando Norris,mclaren,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,6,5,10.0,Finished,2


## jolpica race_schedule data preview raw -> staging

In [35]:
with fs.open(f"s3://{bucket}/jolpica/2026/race_schedule.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,season,round,race_name,circuit,race_date,race_start_time,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,sprint_qualy_date,sprint_qualy_time,qualy_date,qualy_time,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,04:00:00Z,2026-03-06,01:30:00Z,2026-03-06,05:00:00Z,2026-03-07,01:30:00Z,None,None,2026-03-07,05:00:00Z,jolpica,2026-03-18T21:46:42.364626+00:00
1,2026,2,Chinese Grand Prix,Shanghai International Circuit,2026-03-15,07:00:00Z,2026-03-13,03:30:00Z,None,None,None,None,2026-03-13,07:30:00Z,2026-03-14,07:00:00Z,jolpica,2026-03-18T21:46:42.364626+00:00
2,2026,3,Japanese Grand Prix,Suzuka Circuit,2026-03-29,05:00:00Z,2026-03-27,02:30:00Z,2026-03-27,06:00:00Z,2026-03-28,02:30:00Z,None,None,2026-03-28,06:00:00Z,jolpica,2026-03-18T21:46:42.364626+00:00


In [36]:
df = con.execute("SELECT * FROM bronze.jolpica_race_schedule").df()
df.head(3)

,season,round,race_name,circuit,race_date,race_start_time,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,sprint_qualy_date,sprint_qualy_time,qualy_date,qualy_time,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,04:00:00Z,2026-03-06,01:30:00Z,2026-03-06,05:00:00Z,2026-03-07,01:30:00Z,None,None,2026-03-07,05:00:00Z,jolpica,2026-03-18T21:46:42.364626+00:00
1,2026,2,Chinese Grand Prix,Shanghai International Circuit,2026-03-15,07:00:00Z,2026-03-13,03:30:00Z,None,None,None,None,2026-03-13,07:30:00Z,2026-03-14,07:00:00Z,jolpica,2026-03-18T21:46:42.364626+00:00
2,2026,3,Japanese Grand Prix,Suzuka Circuit,2026-03-29,05:00:00Z,2026-03-27,02:30:00Z,2026-03-27,06:00:00Z,2026-03-28,02:30:00Z,None,None,2026-03-28,06:00:00Z,jolpica,2026-03-18T21:46:42.364626+00:00


In [37]:
df = con.execute("SELECT * FROM silver.jolpica_race_schedule").df()
df.head(3)

,season,round,race_name,circuit,source,ingested_at,race_datetime,fp1_datetime,fp2_datetime,fp3_datetime,sprint_qualy_datetime,qualy_datetime
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,jolpica,2026-03-18 11:46:42.364626-10:00,2026-03-07 18:00:00-10:00,2026-03-05 15:30:00-10:00,2026-03-05 19:00:00-10:00,2026-03-06 15:30:00-10:00,NaT,2026-03-06 19:00:00-10:00
1,2026,2,Chinese Grand Prix,Shanghai International Circuit,jolpica,2026-03-18 11:46:42.364626-10:00,2026-03-14 21:00:00-10:00,2026-03-12 17:30:00-10:00,NaT,NaT,2026-03-12 21:30:00-10:00,2026-03-13 21:00:00-10:00
2,2026,3,Japanese Grand Prix,Suzuka Circuit,jolpica,2026-03-18 11:46:42.364626-10:00,2026-03-28 19:00:00-10:00,2026-03-26 16:30:00-10:00,2026-03-26 20:00:00-10:00,2026-03-27 16:30:00-10:00,NaT,2026-03-27 20:00:00-10:00


In [54]:
df = con.execute("SELECT * FROM staging.stg_jolpica_race_schedule").df()
# df.dtypes
df.head(5)

,season,round,race_name,circuit,race_datetime,fp1_datetime,fp2_datetime,fp3_datetime,sprint_qualy_datetime,qualy_datetime,is_sprint_weekend
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-07 18:00:00-10:00,2026-03-05 15:30:00-10:00,2026-03-05 19:00:00-10:00,2026-03-06 15:30:00-10:00,NaT,2026-03-06 19:00:00-10:00,False
1,2026,2,Chinese Grand Prix,Shanghai International Circuit,2026-03-14 21:00:00-10:00,2026-03-12 17:30:00-10:00,NaT,NaT,2026-03-12 21:30:00-10:00,2026-03-13 21:00:00-10:00,True
2,2026,3,Japanese Grand Prix,Suzuka Circuit,2026-03-28 19:00:00-10:00,2026-03-26 16:30:00-10:00,2026-03-26 20:00:00-10:00,2026-03-27 16:30:00-10:00,NaT,2026-03-27 20:00:00-10:00,False
3,2026,4,Miami Grand Prix,Miami International Autodrome,2026-05-03 10:00:00-10:00,2026-05-01 06:30:00-10:00,NaT,NaT,2026-05-01 10:30:00-10:00,2026-05-02 10:00:00-10:00,True
4,2026,5,Canadian Grand Prix,Circuit Gilles Villeneuve,2026-05-24 10:00:00-10:00,2026-05-22 06:30:00-10:00,NaT,NaT,2026-05-22 10:30:00-10:00,2026-05-23 10:00:00-10:00,True


## jolpica pit_stops data preview raw -> staging

In [41]:
with fs.open(f"s3://{bucket}/jolpica/2026/pit_stops/1.parquet", "rb") as f:
    df_results = pd.read_parquet(f)
pd.set_option('display.max_columns', None)
df_results.head(3)

,season,round,race_name,circuit,date,driver_id,lap_pitted,stop_num,pit_in_time,pit_stop_duration,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,colapinto,9,1,15:16:40,27.733,jolpica,2026-03-18T21:57:00.363321+00:00
1,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,norris,11,1,15:19:21,18.266,jolpica,2026-03-18T21:57:00.363321+00:00
2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,ocon,11,1,15:19:26,18.570,jolpica,2026-03-18T21:57:00.363321+00:00


In [42]:
df = con.execute("SELECT * FROM bronze.jolpica_pit_stops").df()
df.head(3)

,season,round,race_name,circuit,date,driver_id,lap_pitted,stop_num,pit_in_time,pit_stop_duration,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,colapinto,9,1,15:16:40,27.733,jolpica,2026-03-18T21:57:00.363321+00:00
1,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,norris,11,1,15:19:21,18.266,jolpica,2026-03-18T21:57:00.363321+00:00
2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,ocon,11,1,15:19:26,18.570,jolpica,2026-03-18T21:57:00.363321+00:00


In [43]:
df = con.execute("SELECT * FROM silver.jolpica_pit_stops").df()
df.head(3)

,season,round,race_name,circuit,date,driver_id,lap_pitted,stop_num,pit_in_time,pit_stop_duration,source,ingested_at
0,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,colapinto,9,1,15:16:40,27.733,jolpica,2026-03-18 11:57:00.363321-10:00
1,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,norris,11,1,15:19:21,18.266,jolpica,2026-03-18 11:57:00.363321-10:00
2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,ocon,11,1,15:19:26,18.570,jolpica,2026-03-18 11:57:00.363321-10:00


In [55]:
df = con.execute("SELECT * FROM staging.stg_jolpica_pit_stops").df()
# df.dtypes
df.head(5)

,season,round,driver_id,race_name,circuit,race_date,pit_stop_number,lap_pitted,pit_in_time,pit_stop_duration_seconds
0,2026,1,colapinto,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,9,15:16:40,27.733
1,2026,1,norris,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,11,15:19:21,18.266
2,2026,1,ocon,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,11,15:19:26,18.570
3,2026,1,gasly,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,11,15:19:27,18.672
4,2026,1,sainz,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,11,15:19:35,19.859


In [57]:
con.close()

# Gold mart data previews

In [86]:
con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))

In [77]:
df = con.execute("SELECT * FROM gold.mart_race_results").df()
# df.dtypes
df.head(3)

,result_id,season,round,driver_id,driver_number,abbreviation,full_name,team_id,team_name,team_color,constructor,race_name,circuit,race_date,is_sprint_weekend,grid_position,finish_position,classified_position,laps,race_time_seconds,points,status,fastest_lap_rank
0,f08a4588c6c8ed01f712ffac933adfc8,2026,1,russell,63,RUS,George Russell,mercedes,Mercedes,00D7B6,mercedes,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,False,1,1,1,58,4986.801,25.0,Finished,6
1,30f8bc62eef74e82f5c22204fc990a18,2026,1,antonelli,12,ANT,Kimi Antonelli,mercedes,Mercedes,00D7B6,mercedes,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,False,2,2,2,58,2.974,18.0,Finished,3
2,ceeb93105bb8543e9cc5cae92e7712fb,2026,1,leclerc,16,LEC,Charles Leclerc,ferrari,Ferrari,ED1131,ferrari,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,False,4,3,3,58,15.519,15.0,Finished,5


In [78]:
df = con.execute("SELECT * FROM gold.mart_constructor_standings ORDER BY championship_points DESC").df()
# df.dtypes
df.head(3)

,standing_id,season,round,constructor_id,constructor,constructor_country,championship_position,championship_points,wins,points_this_round
0,134e79f4a28828cc9fa7b2be800379a3,2026,2,mercedes,Mercedes,German,1,98.0,2,NaN
1,1e499df2bc752d6cee99bcc302e3ba18,2026,2,ferrari,Ferrari,Italian,2,67.0,0,NaN
2,1824d3aed0f0b0e28263782d2e835be3,2026,2,mclaren,McLaren,British,3,18.0,0,NaN


In [79]:
df = con.execute("SELECT * FROM gold.mart_driver_standings ORDER BY championship_points DESC").df()
# df.dtypes
df.head(3)

,standing_id,season,round,driver_id,abbreviation,driver_name,driver_dob,driver_country,constructor,constructor_country,championship_position,championship_points,wins,points_this_round
0,7f23b5ca1092f0ea944925e25ea9820e,2026,2,russell,RUS,George Russell,1998-02-15,British,mercedes,German,1,51.0,1,NaN
1,09f5ff7b7e0a91f681480db17514d948,2026,2,antonelli,ANT,Andrea Kimi Antonelli,2006-08-25,Italian,mercedes,German,2,47.0,1,NaN
2,b51c5aaf93e59f4e17b690386a8554d5,2026,2,leclerc,LEC,Charles Leclerc,1997-10-16,Monegasque,ferrari,Italian,3,34.0,0,NaN


In [80]:
df = con.execute("SELECT * FROM gold.mart_weather").df()
# df.dtypes
df.head(3)

,weather_id,season,round,race_name,circuit,race_datetime,is_sprint_weekend,session_elapsed_seconds,air_temp,track_temp,track_air_temp_delta,humidity,pressure,rainfall,wind_speed,wind_direction
0,6644c8ade57bacd206dff71f192ffd6e,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-07 18:00:00-10:00,False,13.466,23.1,36.2,13.1,55.8,1013.7,False,1.8,114
1,9bc1c553b55b1677f54b0d18f006c9e2,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-07 18:00:00-10:00,False,73.482,23.2,35.9,12.7,55.1,1013.6,False,3.0,110
2,681878e18c1c4effacf8427cfdde237f,2026,1,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-07 18:00:00-10:00,False,133.483,23.2,35.6,12.4,55.2,1013.6,False,3.2,115


In [81]:
df = con.execute("SELECT * FROM gold.mart_lap_times").df()
# df.dtypes
df.head(3)

,lap_id,season,round,abbreviation,driver_id,full_name,team_id,team_name,team_color,lap_number,stint,lap_time_seconds,sector1_time_seconds,sector2_time_seconds,sector3_time_seconds,speed_i1,speed_i2,speed_fl,speed_st,compound,tyre_life,fresh_tyre,is_accurate,is_personal_best,track_status,track_position,pit_out_time_seconds,pit_in_time_seconds
0,0702209e59f1fb8d66b04594375de8b2,2026,1,NOR,norris,Lando Norris,mclaren,McLaren,F47600,1,1,96.458,NaN,18.163,38.796,229.0,291.0,304.0,217.0,MEDIUM,1,True,False,False,1,6,NaN,NaN
1,f879253289ee73c9384682e261be9b7a,2026,1,NOR,norris,Lando Norris,mclaren,McLaren,F47600,2,1,87.344,31.074,18.116,38.154,243.0,287.0,303.0,260.0,MEDIUM,2,True,True,True,1,6,NaN,NaN
2,2961d98eedbf3b93ce19d58c2dad8b56,2026,1,NOR,norris,Lando Norris,mclaren,McLaren,F47600,3,1,86.863,30.541,18.252,38.070,245.0,311.0,286.0,273.0,MEDIUM,3,True,True,True,1,7,NaN,NaN


In [82]:
df = con.execute("SELECT * FROM gold.mart_pit_stops").df()
# df.dtypes
df.head(3)

,pit_stop_id,season,round,driver_id,driver_abbreviation,race_name,circuit,race_date,pit_stop_number,lap_pitted,pit_in_time,pit_stop_duration_seconds,stint_ending,compound_ending,compound_starting,tyre_age_at_pit,pit_lap_time_seconds,pit_lap_is_accurate,is_slow_pit_stop
0,3c2df39f4fe8402e59fad0ccc552e7d9,2026,1,albon,ALB,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,1,12,15:21:28,18.118,1,MEDIUM,HARD,12,125.578,False,False
1,4c5af2cec638438892294d19272395cd,2026,1,albon,ALB,Australian Grand Prix,Albert Park Grand Prix Circuit,2026-03-08,2,33,15:53:04,18.404,2,HARD,None,21,110.030,False,False
2,3d7238c9fc64107ebe8c81b80138f528,2026,2,hulkenberg,HUL,Chinese Grand Prix,Shanghai International Circuit,2026-03-15,1,35,16:04:32,37.358,1,HARD,None,35,104.913,False,True


In [83]:
df = con.execute("SELECT * FROM gold.mart_telemetry_stints").df()
# df.dtypes
df.head(3)

,telemetry_stint_id,season,round,abbreviation,driver_id,full_name,team_id,team_name,team_color,stint,compound,fresh_tyre,stint_laps,tyre_age_start,tyre_age_end,avg_speed,max_speed,min_speed_on_throttle,avg_throttle,full_throttle_pct,brake_pct,drs_open_pct,avg_gear,max_distance_metres
0,bf34f6661a0cbd7ff56b0e40f825c167,2026,2,LEC,leclerc,Charles Leclerc,ferrari,Ferrari,ED1131,1,MEDIUM,True,10,1,10,190.080,333.0,16.471,63.664,50.306,29.273,0.0,4.520,5504.888244
1,e93c0f509bbff185162c0935cda18ff3,2026,1,COL,colapinto,Franco Colapinto,alpine,Alpine,00A1E8,3,SOFT,True,10,1,10,221.265,343.0,79.000,68.219,50.219,17.016,0.0,5.220,5238.196228
2,0c0191910fea7daea33d585500627837,2026,1,SAI,sainz,Carlos Sainz,williams,Williams,1868DB,4,SOFT,True,11,1,11,218.552,326.0,35.000,67.196,50.660,17.046,0.0,5.058,5247.194128


In [87]:
df = con.execute("SELECT * FROM gold.mart_tyre_strategy").df()
# df.dtypes
df.head(3)

,tyre_stint_id,season,round,abbreviation,driver_id,full_name,team_id,team_name,team_color,stint,compound,fresh_tyre,stint_start_lap,stint_end_lap,stint_length_laps,tyre_age_end,avg_lap_time_seconds,fastest_lap_seconds
0,f879253289ee73c9384682e261be9b7a,2026,1,NOR,norris,Lando Norris,mclaren,McLaren,F47600,2,HARD,True,12,34,23,23,83.906353,83.394
1,68446e40026af47eba7ad1cd2feae248,2026,1,LAW,lawson,Liam Lawson,rb,Racing Bulls,6C98FF,2,HARD,True,12,33,22,22,85.749412,84.910
2,1e5c2f2913b4ba62e28c8fe4cd4cf328,2026,1,LIN,arvid_lindblad,Arvid Lindblad,rb,Racing Bulls,6C98FF,1,MEDIUM,True,1,18,18,18,85.636750,84.877


In [7]:
con.close()